# StreamClipper-Style Hook & Payoff Generator
Generates vertical `1080x1920` Shorts (**25s?50s**, hook + full payoff?no 13s dead clips) saved to `MyDrive/StreamClipper/` with `clips.csv`, `clips.json`, and `report/index.html`.
1. **Runtime > Change runtime type > T4 GPU**.
2. Run cells top to bottom; fill in the form fields and run.

In [ ]:
#@title 1. Check GPU & Mount Google Drive
from google.colab import drive
import os, subprocess

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/StreamClipper', exist_ok=True)
subprocess.run(['nvidia-smi'], check=False)
print('Drive folder ready: /content/drive/MyDrive/StreamClipper')

In [ ]:
#@title 2. Clone Repo & Install Dependencies (No Paid APIs Required)
import os, shutil, subprocess, time

!curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 || true
os.environ['PATH'] += ':/root/.deno/bin:/usr/local/bin'
!rm -rf hype-clipper && git clone https://github.com/Aromalajith8/hype-clipper.git hype-clipper
!pip install -q -r hype-clipper/requirements.txt
print('Dependencies installed!')

In [ ]:
#@title 3. Stream, Hook/Payoff Length (<= 50s) & Free AI Settings
STREAM_URL = '' #@param {type:"string"}
CLIPS = 40 #@param {type:"slider", min:5, max:100, step:5}
MIN_LEN = 25 #@param {type:"slider", min:15, max:45, step:5}
MAX_LEN = 50 #@param {type:"slider", min:20, max:50, step:5}
CHAT_BOX_SIDE = 'right' #@param ['right', 'left', 'none']
SPLIT_CAM = False #@param {type:"boolean"}
AI_PROVIDER = 'auto' #@param ['auto', 'groq', 'gemini', 'ollama', 'heuristic']
OLLAMA_MODEL = 'qwen3.5:9b' #@param ['qwen3.5:9b', 'gemma4:12b']
GROQ_API_KEY = '' #@param {type:"string"}
GEMINI_API_KEY = '' #@param {type:"string"}
COOKIES_PATH = '/content/drive/MyDrive/cookies.txt' #@param {type:"string"}

In [ ]:
#@title 4. Run Clipper (Saves to MyDrive/StreamClipper & Resumes Automatically)
import os, shutil, subprocess, time

assert STREAM_URL.strip(), 'Paste a stream URL in STREAM_URL inside Step 3 first!'

if GROQ_API_KEY.strip():
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY.strip()
if GEMINI_API_KEY.strip():
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY.strip()

if AI_PROVIDER == 'ollama' or (AI_PROVIDER == 'auto' and not GROQ_API_KEY.strip() and not GEMINI_API_KEY.strip()):
    if not shutil.which('ollama'):
        !apt-get -qq update && apt-get -qq install -y zstd curl > /dev/null 2>&1 || true
        !curl -fsSL -L https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tgz -o /tmp/ollama.tgz
        !mkdir -p /usr/local && tar -xzf /tmp/ollama.tgz -C /usr/local && rm -f /tmp/ollama.tgz
    ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
    subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    subprocess.run([ollama_bin, 'pull', OLLAMA_MODEL], check=False)

cmd = [
    'python3', 'hype-clipper/clipper.py', STREAM_URL.strip(),
    '--clips', str(CLIPS),
    '--min-len', str(MIN_LEN),
    '--max-len', str(min(MAX_LEN, 50)),
    '--chat-box', CHAT_BOX_SIDE,
    '--out', '/content/drive/MyDrive/StreamClipper',
    '--ai-provider', AI_PROVIDER,
    '--ollama-model', OLLAMA_MODEL,
]

if SPLIT_CAM:
    cmd.append('--split-cam')
if os.path.exists(COOKIES_PATH):
    cmd += ['--cookies', COOKIES_PATH]

subprocess.run(cmd)